# TTF Options — Dashboard Jupyter

Dashboard interactif pour le pricing des options TTF (gaz naturel europeen), base sur `black76_ttf.py`.

## Section 1 - Pricer

Reglez les parametres ci-dessous. Les prix Call / Put et les Greeks se recalculent automatiquement.

**Conventions**
- Forward, Strike : EUR/MWh
- Maturite : annees (ACT/365)
- Taux : decimal annualise (`0.03` = 3 %)
- Vol Black-76 : lognormale decimale (`0.50` = 50 %)
- Vol Bachelier : normale en EUR/MWh

In [ ]:
import ipywidgets as widgets
from IPython.display import display

import black76_ttf as b76m

In [ ]:
# ---- Widgets ----
style = {"description_width": "140px"}
layout = widgets.Layout(width="480px")

model = widgets.Dropdown(
    options=["Black-76", "Bachelier"], value="Black-76",
    description="Modele :", style=style, layout=layout,
)
forward = widgets.FloatSlider(
    value=35.0, min=1.0, max=150.0, step=0.5,
    description="Forward (EUR/MWh) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
strike = widgets.FloatSlider(
    value=35.0, min=1.0, max=150.0, step=0.5,
    description="Strike (EUR/MWh) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
maturity = widgets.FloatSlider(
    value=0.25, min=0.01, max=3.0, step=0.01,
    description="Maturite (annees) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
rate = widgets.FloatSlider(
    value=0.03, min=-0.02, max=0.10, step=0.0025,
    description="Taux (decimal) :", style=style, layout=layout,
    continuous_update=False, readout_format=".4f",
)
vol_b76 = widgets.FloatSlider(
    value=0.50, min=0.01, max=2.00, step=0.01,
    description="Vol (lognormale) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
vol_bach = widgets.FloatSlider(
    value=10.0, min=0.1, max=50.0, step=0.1,
    description="Vol (EUR/MWh) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
vol_bach.layout.display = "none"  # hidden until Bachelier is selected

output = widgets.Output()


def _fmt_greeks(g) -> str:
    return (
        f"  Delta : {g.delta:+.4f}\n"
        f"  Gamma : {g.gamma:+.6f}\n"
        f"  Vega  : {g.vega:+.4f}\n"
        f"  Theta : {g.theta:+.4f} / jour\n"
        f"  Rho   : {g.rho:+.4f} / pp\n"
        f"  Vanna : {g.vanna:+.6f}\n"
        f"  Volga : {g.volga:+.6f}"
    )


def _recompute(*_):
    if model.value == "Black-76":
        vol_b76.layout.display = ""
        vol_bach.layout.display = "none"
    else:
        vol_b76.layout.display = "none"
        vol_bach.layout.display = ""

    F = forward.value
    K = strike.value
    T = maturity.value
    r = rate.value

    with output:
        output.clear_output()
        try:
            if model.value == "Black-76":
                sigma = vol_b76.value
                call = b76m.b76_call(F, K, T, r, sigma)
                put = b76m.b76_put(F, K, T, r, sigma)
                g_call = b76m.b76_greeks(F, K, T, r, sigma, "call")
                g_put = b76m.b76_greeks(F, K, T, r, sigma, "put")
                vol_label = f"sigma = {sigma:.2%} (lognormale)"
            else:
                sigma_n = vol_bach.value
                call = b76m.bach_call(F, K, T, r, sigma_n)
                put = b76m.bach_put(F, K, T, r, sigma_n)
                g_call = b76m.bach_greeks(F, K, T, r, sigma_n, "call")
                g_put = b76m.bach_greeks(F, K, T, r, sigma_n, "put")
                vol_label = f"sigma_n = {sigma_n:.2f} EUR/MWh (normale)"

            print(f"=== {model.value} - {vol_label} ===")
            print(f"F = {F:.2f}   K = {K:.2f}   T = {T:.4f} y   r = {r:.4f}")
            print("")
            print(f"Prix Call : {call:.4f} EUR/MWh")
            print(f"Prix Put  : {put:.4f} EUR/MWh")
            print("")
            print("Greeks Call :")
            print(_fmt_greeks(g_call))
            print("")
            print("Greeks Put :")
            print(_fmt_greeks(g_put))
        except Exception as exc:
            print(f"Erreur de calcul : {exc}")


for w in (model, forward, strike, maturity, rate, vol_b76, vol_bach):
    w.observe(_recompute, names="value")

_recompute()

display(widgets.VBox([
    model, forward, strike, maturity, rate, vol_b76, vol_bach,
    widgets.HTML("<hr>"),
    output,
]))